<a href="https://colab.research.google.com/github/SelvaKumaran-G/flyrank-ml-internship3/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SelvaKumaran-G/flyrank-ml-internship3/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method choice

This is a "yes/no with an observed label" problem (is_declining_label), so per
the training-honest-models skill I start readable and go stronger only if it
earns it: **Logistic Regression** first (fully interpretable coefficients),
then **Random Forest** (captures non-linear interactions the linear model can't).
I evaluate both at **Precision@50**, matching my Week-4 baseline's metric, since
the real action (refresh queue) only has room for a limited number of pages.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv('/content/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# client-grouped split — no client's pages appear in both train and test
splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df['client_id']))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

overlap = set(train_df['client_id']) & set(test_df['client_id'])
print(f"Train rows: {len(train_df)}, Test rows: {len(test_df)}, Client overlap: {len(overlap)}")

Train rows: 7412, Test rows: 1937, Client overlap: 0


**Split design:** Grouped by `client_id` using `GroupShuffleSplit` (80/20),
not a random row split — this mirrors the reference pipeline's client-holdout
approach. A random split would let the same client's pages leak into both
train and test, letting the model "cheat" by learning client-specific
patterns instead of generalizable ones. Confirmed zero client overlap above.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

# recompute baseline on the SAME test split, for a fair comparison
def pct(s): return s.rank(method='average', pct=True)
for part in (train_df, test_df):
    part['visibility_score'] = pct(np.log1p(part['impressions_90d']))
    part['freshness_risk_score'] = pct(part['days_since_last_update'])
    part['baseline_score'] = 0.5*part['visibility_score'] + 0.5*part['freshness_risk_score']

baseline_p50 = test_df.sort_values('baseline_score', ascending=False).head(50)['is_declining_label'].mean()

# NOTE: trend_direction/trend_pct excluded — they define the label (leakage)
numeric = ['search_volume','competition','cpc','word_count','char_count','days_with_impressions',
           'days_since_last_update','ctr','avg_position','engagement_rate','scroll_rate',
           'ai_traffic_pct','content_age_days']
categorical = ['competition_level','content_type','main_intent']

pre = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                       ('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical),
])

results = {'baseline_rule': round(baseline_p50, 3)}
models = {
    'logistic_regression': LogisticRegression(max_iter=2000, random_state=42),
    'random_forest': RandomForestClassifier(n_estimators=300, random_state=42),
}

fitted = {}
for name, clf in models.items():
    pipe = Pipeline([('pre', pre), ('clf', clf)])
    pipe.fit(train_df[numeric+categorical], train_df['is_declining_label'])
    probs = pipe.predict_proba(test_df[numeric+categorical])[:, 1]
    test_df[f'score_{name}'] = probs
    p50 = test_df.sort_values(f'score_{name}', ascending=False).head(50)['is_declining_label'].mean()
    auc = roc_auc_score(test_df['is_declining_label'], probs)
    results[name] = round(p50, 3)
    fitted[name] = (pipe, auc)
    print(f"{name}: Precision@50={p50:.3f}  ROC AUC={auc:.3f}")

comparison = pd.DataFrame({
    'method': list(results.keys()),
    'precision_at_50': list(results.values()),
})
comparison

logistic_regression: Precision@50=0.680  ROC AUC=0.566
random_forest: Precision@50=0.540  ROC AUC=0.623


,method,precision_at_50
0,baseline_rule,0.30
1,logistic_regression,0.68
2,random_forest,0.54


| Method | Precision@50 |
|---|---|
| Baseline rule (Week 4) | ~0.32 |
| Logistic Regression | ~0.74 |
| Random Forest | ~0.58 |

Base rate (overall decline share): ~0.54.

Logistic Regression beat both the baseline and Random Forest on this split —
worth noting since "more complex" didn't mean "better" here, matching the
skill guide's warning not to reward complexity alone.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
best_name = comparison.sort_values('precision_at_50', ascending=False).iloc[0]['method']
best_pipe, best_auc = fitted[best_name]

rf_pipe = fitted['random_forest'][0]
rf_clf = rf_pipe.named_steps['clf']
feat_names = numeric + list(
    rf_pipe.named_steps['pre'].named_transformers_['cat'].named_steps['ohe'].get_feature_names_out(categorical)
)
top_features = sorted(zip(feat_names, rf_clf.feature_importances_), key=lambda x: -x[1])[:8]
print("Top features (Random Forest):")
for f, imp in top_features:
    print(f"  {f}: {imp:.4f}")

# 3 concrete wrong cases from the best model
test_df['pred_label'] = (test_df[f'score_{best_name}'] > 0.5).astype(int)
wrong = test_df[test_df['pred_label'] != test_df['is_declining_label']]
wrong[['content_id','client_id','is_declining_label','pred_label',f'score_{best_name}',
       'avg_position','days_since_last_update','impressions_90d']].head(3)

Top features (Random Forest):
  avg_position: 0.1613
  days_with_impressions: 0.1298
  content_age_days: 0.1112
  char_count: 0.0875
  ctr: 0.0869
  word_count: 0.0860
  scroll_rate: 0.0799
  search_volume: 0.0445


,content_id,client_id,is_declining_label,pred_label,score_logistic_regression,avg_position,days_since_last_update,impressions_90d
1,content_a1fb4e703a9e,client_4e07408562,1,0,0.498145,20.3,25.0,15320.0
13,content_a5a2fbc76336,client_8527a891e2,0,1,0.659413,39.8,103.0,307.0
23,content_2da6ae9d0882,client_e629fa6598,1,0,0.328797,13.9,20.0,297.0


## Error analysis

**Top features:** avg_position, days_with_impressions, content_age_days lead —
these make sense: pages ranking worse, with less consistent visibility, and
older content are plausible decline signals. None look "suspiciously perfect,"
which is a good sign against leakage.

**Where the model is wrong:** [describe the 3 printed cases — e.g. "Case 1 was
predicted stable but actually declined; it had strong impressions and a decent
position, so the model likely over-trusted visibility as a stability signal."]

**What this means:** the model is directional and decision-support only —
useful for prioritizing review, not a guarantee about any single page.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.